# Week 3 – CUAD PDF/TXT Data Pipeline
**Project:** AI-Powered Contract and Terms & Conditions Analyzer  
**Dataset:** CUAD v1 (user-provided ZIP archive)  
**Pathway:** PDF/TXT document ingestion  

Run cells in order. This notebook performs workspace setup, archive ingestion, recursive verification, source inspection, cleaning, chunking, export, validation, lineage and repeatability checks. It processes one matched PDF/TXT pair from the same agreement; these are two formats of one contract, not two independent contracts. It does not train a model or evaluate clause-detection accuracy.

In Colab, upload the CUAD ZIP when prompted, or set ZIP_PATH to a file in mounted Google Drive.

## A2 – Required workspace setup

In [1]:
from pathlib import Path
import json, hashlib, re, zipfile, statistics
from datetime import datetime, timezone
BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW, BASE, OUT, PROCESSED = RAW_DIR, BASE_DIR, PROCESSED_DIR, PROCESSED_DIR
print('BASE_DIR:', BASE_DIR); print('RAW_DIR:', RAW_DIR); print('PROCESSED_DIR:', PROCESSED_DIR)

BASE_DIR: /content/week3_project
RAW_DIR: /content/week3_project/data/raw
PROCESSED_DIR: /content/week3_project/data/processed


## A2 – Ingest actual CUAD data

In [2]:
# Set this if the ZIP is already in Drive; otherwise leave as None and upload it below.
ZIP_PATH = None  # Example: '/content/drive/MyDrive/CUAD_v1.zip'
if ZIP_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zip_names: raise FileNotFoundError('Upload the CUAD ZIP archive.')
    ZIP_PATH = Path('/content') / zip_names[0]
else: ZIP_PATH = Path(ZIP_PATH)
if not ZIP_PATH.is_file(): raise FileNotFoundError(ZIP_PATH)
CONTRACT_STEM = '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement'
with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    pdf_member = next((n for n in names if '/full_contract_pdf/' in n and n.endswith('/' + CONTRACT_STEM + '.pdf')), None)
    txt_member = next((n for n in names if '/full_contract_txt/' in n and n.endswith('/' + CONTRACT_STEM + '.txt')), None)
    if not pdf_member or not txt_member: raise FileNotFoundError('Matched CUAD PDF/TXT pair not found in archive.')
    pdf_path = RAW_DIR / (CONTRACT_STEM + '.pdf'); txt_path = RAW_DIR / (CONTRACT_STEM + '.txt')
    pdf_path.write_bytes(z.read(pdf_member)); txt_path.write_bytes(z.read(txt_member))
print('Archive:', ZIP_PATH.name, '| bytes:', ZIP_PATH.stat().st_size)
print('Extracted:', pdf_path.name, 'and', txt_path.name)

Saving CUAD_v1 (2).zip to CUAD_v1 (2).zip
Archive: CUAD_v1 (2).zip | bytes: 105883672
Extracted: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.pdf and 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt


## A2 – Required recursive input verification

In [3]:
input_files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(input_files))
for p in input_files: print(p.relative_to(RAW_DIR), p.stat().st_size, 'bytes')
assert pdf_path in input_files and txt_path in input_files
print('Input verification: PASS')

Input count: 2
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.pdf 109658 bytes
2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt 29454 bytes
Input verification: PASS


## A3 – Inspect at least two representative source files

In [4]:
!pip -q install pypdf
from pypdf import PdfReader
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
pages = [(i+1, page.extract_text() or '') for i,page in enumerate(PdfReader(str(pdf_path)).pages)]
pdf_text = '\n'.join(t for _,t in pages)
txt_text = txt_path.read_text(encoding='utf-8', errors='strict')
assert len(pdf_text.strip()) > 1000 and len(txt_text.strip()) > 1000
sources = [(pdf_path, pdf_text, 'PDF', len(pages)), (txt_path, txt_text, 'TXT', 1)]
inspection=[]
for p, text, fmt, units in sources:
    row={'filename':p.name,'format':fmt,'size_bytes':p.stat().st_size,'pages_or_segments':units,'extracted_characters':len(text),'sha256':sha256_file(p)}
    inspection.append(row); print(row); print('Sample:', repr(text[:220]))
print('Source inspection: PASS')

{'filename': '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.pdf', 'format': 'PDF', 'size_bytes': 109658, 'pages_or_segments': 9, 'extracted_characters': 29372, 'sha256': 'db445255a65067160b9e9a7cf23afbd3379a715d000b0576fa8e7f64982e523c'}
Sample: '                CO-BRANDING AND ADVERTISING AGREEMENT\nTHIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made\nas of June 21, 1999 (the "Effective Date") by and between I-ESCROW,\nINC., with its principal plac'
{'filename': '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt', 'format': 'TXT', 'size_bytes': 29454, 'pages_or_segments': 1, 'extracted_characters': 29454, 'sha256': '6cdfab95eb5b30c0d2d4860c8204b96b36a54cb9e2025c5ceb1457d223afac41'}
Sample: 'CO-BRANDING AND ADVERTISING AGREEMENT\n\nTHIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made as of June 21, 1999 (the "Effective Date") by and between I-ESCROW, IN

## B2/B3 – Clean, compare chunking, and prepare the PDF/TXT pathway

In [14]:
FIXED_SIZE, MAX_CHUNK, MIN_CHUNK = 1000, 3000, 80
def clean_text(s):
    s=s.replace('\r\n','\n').replace('\r','\n').replace('\x0c','\n')
    s=re.sub(r'[ \t]+',' ',s); s=re.sub(r'\n{3,}','\n\n',s); return s.strip()
def fixed_chunks(s,size=FIXED_SIZE): return [s[i:i+size].strip() for i in range(0,len(s),size) if s[i:i+size].strip()]
HEADING=re.compile(r'^\s*(ARTICLE\s+[IVXLC\d]+\b.*|SECTION\s+\d+(\.\d+)*\b.*|Section\s+\d+(\.\d+)*\b.*|\d{1,2}(\.\d{1,2})*\.?\s+[A-Z][A-Za-z ,&/\-]{2,60}\.?\s*)$')
def split_long(s,limit=MAX_CHUNK):
    if len(s)<=limit: return [s]
    parts=[]; cur=''
    for para in re.split(r'\n\s*\n',s):
        if cur and len(cur)+len(para)+2>limit: parts.append(cur.strip()); cur=para
        else: cur=(cur+'\n\n'+para) if cur else para
    if cur.strip(): parts.append(cur.strip())
    out=[]
    for p in parts: out.extend([p] if len(p)<=limit else fixed_chunks(p,limit))
    return out
def structure_chunks(s):
    parts=[]; cur=[]
    for line in s.splitlines():
        if HEADING.match(line) and any(x.strip() for x in cur): parts.append('\n'.join(cur).strip()); cur=[line]
        else: cur.append(line)
    if any(x.strip() for x in cur): parts.append('\n'.join(cur).strip())
    if len(parts)<3: parts=[x.strip() for x in re.split(r'\n\s*\n',s) if x.strip()]
    merged=[]; carry=''
    for p in parts:
        p=(carry+'\n'+p).strip() if carry else p
        if len(p)<MIN_CHUNK: carry=p
        else: merged.append(p); carry=''
    if carry:
        if merged: merged[-1]+='\n'+carry
        else: merged=[carry]
    out=[]
    for p in merged: out.extend(split_long(p))
    return out
prepared={}; comparison=[]; records=[]
for p, raw, fmt, units in sources:
    cleaned=clean_text(raw); fixed=fixed_chunks(cleaned); structured=structure_chunks(cleaned)
    prepared[p.name]=structured
    comparison.append({'filename':p.name,'fixed_chunks':len(fixed),'structure_aware_chunks':len(structured),'fixed_lowercase_start_proxy':sum(1 for x in fixed if x[:1].islower()),'structure_heading_starts':sum(1 for x in structured if x.splitlines() and HEADING.match(x.splitlines()[0]))})
    for i,chunk in enumerate(structured,1): records.append({'source_file':p.name,'source_sha256':sha256_file(p),'chunk_id':f'{p.name}-{i:03d}','chunk_index':i,'method':'structure_aware','char_count':len(chunk),'text':chunk})
print('Chunk comparison:', comparison); print('Total chunks:',len(records))

Chunk comparison: [{'filename': '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.pdf', 'fixed_chunks': 29, 'structure_aware_chunks': 18, 'fixed_lowercase_start_proxy': 23, 'structure_heading_starts': 12}, {'filename': '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt', 'fixed_chunks': 29, 'structure_aware_chunks': 15, 'fixed_lowercase_start_proxy': 22, 'structure_heading_starts': 7}]
Total chunks: 33


## B3 – Decision\n\nStructure-aware chunking is selected because it attempts to keep clause/section headings with their text. Fixed-size 1,000-character chunks are retained as a baseline. The lowercase-start count is only a rough diagnostic, not a definitive sentence-boundary measure. Rule-based heading detection may miss unusual formats.

In [15]:
prepared_path=PROCESSED_DIR/'prepared_text.jsonl'
with prepared_path.open('w',encoding='utf-8') as f:
    for row in records: f.write(json.dumps(row,ensure_ascii=False)+'\n')
exported=[json.loads(line) for line in prepared_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert exported==records and all(r['text'].strip() for r in exported)
print('Prepared output:',prepared_path,'| bytes:',prepared_path.stat().st_size,'| records:',len(exported))
print('JSONL read-back and non-empty chunk checks: PASS')

Prepared output: /content/week3_project/data/processed/prepared_text.jsonl | bytes: 71839 | records: 33
JSONL read-back and non-empty chunk checks: PASS


## B4 – Lineage, manifest and repeatability

In [16]:
processing='Whitespace/newline cleanup; structure-aware heading/paragraph chunking; merge fragments <80 chars; cap chunks at 3000 chars; fixed 1000-character baseline.'
manifest={'project':'AI-Powered Contract and Terms & Conditions Analyzer','topic':'Topic 3 data ingestion and preparation','dataset':'CUAD v1','selected_pathway':'PDF/TXT documents','generated_at_utc':datetime.now(timezone.utc).isoformat(),'archive_filename':ZIP_PATH.name,'archive_size_bytes':ZIP_PATH.stat().st_size,'archive_sha256':sha256_file(ZIP_PATH),'source_contract_pair':CONTRACT_STEM,'sources':[],'prepared_output':{'path':'data/processed/prepared_text.jsonl','size_bytes':prepared_path.stat().st_size,'sha256':sha256_file(prepared_path),'record_count':len(exported)},'chunk_comparison':comparison,'validation':{'input_verification':'PASS','pdf_extraction':'PASS','txt_utf8':'PASS','jsonl_reload':'PASS','non_empty_chunks':'PASS','repeatability':'pending','checksum_lineage':'PASS'},'limitations':['PDF and TXT are two formats of one agreement, not two independent contracts.','CUAD commercial contracts may not represent consumer website T&Cs.','No OCR, multilingual processing, model training or clause-detection accuracy evaluation was performed.','Check individual contract redistribution rights before sharing source or extracted text.']}
for i,(p,raw,fmt,units) in enumerate(sources,1): manifest['sources'].append({'source_id':f'S{i:02d}','source_filename':p.name,'format':fmt,'size_bytes':p.stat().st_size,'sha256':sha256_file(p),'prepared_output':'data/processed/prepared_text.jsonl','prepared_sha256':sha256_file(prepared_path),'processing':processing})
for p,raw,fmt,units in sources: assert structure_chunks(clean_text(raw))==prepared[p.name]
manifest['validation']['repeatability']='PASS'
manifest_path=BASE_DIR/'data'/'week3_manifest.json'; manifest_path.parent.mkdir(parents=True,exist_ok=True)
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
saved=json.loads(manifest_path.read_text(encoding='utf-8'))
for row in saved['sources']:
    src=RAW_DIR/row['source_filename']; out=BASE_DIR/row['prepared_output']
    assert sha256_file(src)==row['sha256'] and sha256_file(out)==row['prepared_sha256']
print('Repeatability: PASS'); print('Source/output checksum lineage: PASS'); print('Manifest:',manifest_path)

Repeatability: PASS
Source/output checksum lineage: PASS
Manifest: /content/week3_project/data/week3_manifest.json


## Data handling and limitations\n\nThe prepared JSONL contains extracted contract text. Keep it local/private unless redistribution rights for the individual underlying contract have been checked. The manifest contains provenance and checksums, not contract excerpts. This pipeline prepares text for later NLP work; it does not train a classifier or establish model performance.

In [17]:
import json
from pathlib import Path
from collections import Counter

path = Path("/content/week3_project/data/processed/prepared_text.jsonl")

# If the Colab session was reset, upload your JSONL file first.
if not path.exists():
    from google.colab import files
    uploaded = files.upload()
    path = Path(next(iter(uploaded)))

records = []
with open(path, "r", encoding="utf-8") as file:
    for line in file:
        records.append(json.loads(line))

print("LAB 4 - DATA QUALITY AUDIT")
print("Total records:", len(records))

# Rule V1: Required fields and valid character counts
required = [
    "source_file", "source_sha256", "chunk_id",
    "chunk_index", "method", "char_count", "text"
]

v1_pass = 0

for record in records:
    if (
        all(field in record for field in required)
        and isinstance(record["text"], str)
        and record["text"].strip() != ""
        and record["char_count"] == len(record["text"])
    ):
        v1_pass += 1

print("\nV1 - Structural integrity")
print("PASS:", v1_pass)
print("FAIL:", len(records) - v1_pass)

# Rule V2: Unique chunk IDs
ids = [record.get("chunk_id") for record in records]
unique_ids = len(set(ids))

print("\nV2 - Unique chunk IDs")
print("Unique IDs:", unique_ids)
print("Duplicate occurrences:", len(ids) - unique_ids)
print("Result:", "PASS" if unique_ids == len(ids) else "FAIL")

# Category coverage (keyword proxy, not verified labels)
payment_words = ["payment", "fee", "fees", "compensation", "invoice"]
termination_words = [
    "terminate", "termination", "expire",
    "expiration", "renewal", "cancel"
]

payment_count = 0
termination_count = 0

for record in records:
    content = record["text"].lower()

    if any(word in content for word in payment_words):
        payment_count += 1

    if any(word in content for word in termination_words):
        termination_count += 1

print("\nCATEGORY COVERAGE")
print("Payment/fees keyword matches:", payment_count)
print("Termination/renewal keyword matches:", termination_count)

# Inspect 15 records across the dataset
sample_positions = [1, 2, 5, 7, 8, 9, 10, 11, 13, 15, 19, 24, 28, 29, 33]

print("\nREPRESENTATIVE SAMPLE")
for position in sample_positions:
    if position <= len(records):
        record = records[position - 1]
        print(
            "Record:", position,
            "| Source:", Path(record["source_file"]).suffix,
            "| Characters:", record["char_count"],
            "| Preview:", record["text"][:100].replace("\n", " ")
        )

LAB 4 - DATA QUALITY AUDIT
Total records: 33

V1 - Structural integrity
PASS: 33
FAIL: 0

V2 - Unique chunk IDs
Unique IDs: 33
Duplicate occurrences: 0
Result: PASS

CATEGORY COVERAGE
Payment/fees keyword matches: 13
Termination/renewal keyword matches: 14

REPRESENTATIVE SAMPLE
Record: 1 | Source: .pdf | Characters: 437 | Preview: CO-BRANDING AND ADVERTISING AGREEMENT THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") i
Record: 2 | Source: .pdf | Characters: 2749 | Preview: 1. DEFINITIONS. (a) "CONTENT" means all content or information, in any medium, provided by a party t
Record: 5 | Source: .pdf | Characters: 457 | Preview: 2.5 SERVICE PERFORMANCE OF INFORMATION TRANSFER MECHANISM. The parties each shall in good faith work
Record: 7 | Source: .pdf | Characters: 1850 | Preview: 5. PAYMENT. 5.1 ADVERTISING FEES. After the Launch Date, i-Escrow shall pay 2TheMart advertising fee
Record: 8 | Source: .pdf | Characters: 3000 | Preview: 6. RIGHTS AND STANDARDS. Source: 2THEMART C

In [18]:
from pathlib import Path

for record in records:
    source_type = Path(record["source_file"]).suffix.upper().replace(".", "")
    record["chunk_id"] = source_type + "-" + str(record["chunk_index"]).zfill(3)

# Check V2 again
ids = [record["chunk_id"] for record in records]

print("V2 - Unique Chunk IDs (After Fix)")
print("Total records:", len(ids))
print("Unique IDs:", len(set(ids)))
print("Duplicate occurrences:", len(ids) - len(set(ids)))
print("Result:", "PASS" if len(ids) == len(set(ids)) else "FAIL")

V2 - Unique Chunk IDs (After Fix)
Total records: 33
Unique IDs: 33
Duplicate occurrences: 0
Result: PASS


In [19]:
import json
from pathlib import Path

output = Path("/content/week3_project/data/processed/prepared_text.jsonl")
output.parent.mkdir(parents=True, exist_ok=True)

with open(output, "w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Corrected dataset saved!")

Corrected dataset saved!


In [20]:
import hashlib
import json

manifest_path = Path("/content/week3_project/data/week3_manifest.json")

with open(manifest_path, "r", encoding="utf-8") as file:
    manifest = json.load(file)

file_bytes = output.read_bytes()

manifest["prepared_output"]["size_bytes"] = len(file_bytes)
manifest["prepared_output"]["sha256"] = hashlib.sha256(file_bytes).hexdigest()
manifest["prepared_output"]["record_count"] = len(records)

for source in manifest["sources"]:
    source["prepared_sha256"] = manifest["prepared_output"]["sha256"]

manifest["week4_correction"] = {
    "issue": "Duplicate chunk IDs across PDF and TXT",
    "action": "Added source format and chunk index to identifiers",
    "v2_result": "PASS" if len(ids) == len(set(ids)) else "FAIL"
}

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Manifest updated")

Manifest updated
